# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata object
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Since Croissant datasets structure data into one or more **record sets** (tables, files, or granular data units), referencing each by its `@id`, we inspect the dataset for available record sets and detail their fields and columns by `@id`.

In [ ]:
# List all Record Sets and their IDs
record_sets = [rs for rs in dataset.metadata.record_sets]
print("Available RecordSets (by @id and name):")
for rs in record_sets:
    print(f"  @id: {rs.id} | name: {rs.name}")

if not record_sets:
    print("No record sets found: attempting to extract from main dataset distribution.")
    # In some Croissant datasets, the data comes as primary files referenced in `distribution`.
    # Let's show distributions as well.
    if hasattr(meta, 'distributions') and meta.distributions:
        for dist in meta.distributions:
            print(f"Distribution: @id={dist.id}, name={dist.name if hasattr(dist, 'name') else '(no name)'}, mediaType={dist.media_type}")
    else:
        print("No distributions found. Please refer to the dataset documentation for data locations.")
else:
    # For each record set, list its fields and columns
    for rs in record_sets:
        print(f"\nFields for RecordSet '{rs.name}' (@id: {rs.id}):")
        for field in rs.fields:
            print(f"  Field @id: {field.id}, name: {field.name}, data_type: {field.data_type}")
            # List columns for the field, if columns exist
            if hasattr(field, 'columns') and field.columns:
                for col in field.columns:
                    print(f"    Column @id: {col.id}, name: {col.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. The record set and field `@id`s from the overview guide which objects to extract.

If the dataset has no explicit RecordSets, we attempt to extract the primary table from the first available distribution (`@id`).

In [ ]:
dataframes = {}

# Default: try to use record set, fallback to distribution (for single-table datasets)
if record_sets:
    record_sets_ids = [rs.id for rs in record_sets]
else:
    # Try loading data from the main distribution
    if hasattr(meta, 'distributions') and meta.distributions:
        record_sets_ids = [meta.distributions[0].id]  # We treat distributions as record sets if no explicit RecordSet
        print(f"Using distribution @id: {record_sets_ids[0]}")
    else:
        raise RuntimeError("No record sets or distributions found in the metadata.")

for record_set in record_sets_ids:
    try:
        # Streaming records from the RecordSet
        records = list(dataset.records(record_set=record_set))
        if len(records) == 0:
            print(f"No records found for RecordSet {record_set}, skipping.")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} records for {record_set}.")
    except Exception as exc:
        print(f"Error loading records for {record_set}: {exc}")

# Show columns for the first successfully loaded dataframe
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns for record set '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Choose a numeric field and a group-by field for demonstration. All referencing is done by `@id`.

In [ ]:
import numpy as np

# You may need to adjust field IDs according to what's printed above in section 2
# For demonstration, let's attempt to guess the primary numeric and group-by fields.

if dataframes:
    df = list(dataframes.values())[0]
    rs_id = list(dataframes.keys())[0]
    # Try to find a numeric column (commonly named 'MW', 'abundance', 'coverage', etc.)
    possible_numeric = [col for col in df.columns if df[col].dtype.kind in 'fi' or 'abundance' in col.lower() or 'mw' in col.lower() or 'coverage' in col.lower()]
    if possible_numeric:
        numeric_field = possible_numeric[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        numeric_field = df.select_dtypes(include=np.number).columns[0] if len(df.select_dtypes(include=np.number).columns) > 0 else df.columns[0]
        print(f"No obvious numeric field, defaulting to: {numeric_field}")

    # Try to choose a group field
    possible_group_field = None
    for col in df.columns:
        if 'group' in col.lower() or 'sample' in col.lower() or 'description' in col.lower():
            possible_group_field = col
            break
    group_field = possible_group_field if possible_group_field else df.columns[0]

    # Filtering
    try:
        numeric_threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 10
        filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > numeric_threshold]
        print(f"Filtered records with {numeric_field} > {numeric_threshold:.2f} (using column '@id': {numeric_field}):")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field], errors='coerce') - pd.to_numeric(filtered_df[numeric_field], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field], errors='coerce').std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
            print(f"Grouped data by '{group_field}':")
            print(grouped_df.head())
    except Exception as e:
        print(f"EDA error: {e}\nCheck that the field types and names are appropriate for your dataset.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn. Adjust the field references as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = list(dataframes.values())[0]
    # Use the same numeric_field and group_field selected above if present
    try:
        if numeric_field in df.columns:
            plt.figure(figsize=(8,4))
            sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
            plt.title(f"Distribution of {numeric_field}")
            plt.xlabel(numeric_field)
            plt.ylabel("Count")
            plt.show()

        if group_field in df.columns and numeric_field in df.columns:
            plt.figure(figsize=(10,5))
            # Only show top 20 groups for better readability
            top_groups = df[group_field].value_counts().index[:20]
            sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=pd.to_numeric(df[numeric_field], errors='coerce'))
            plt.xticks(rotation=90)
            plt.title(f"{numeric_field} by {group_field}")
            plt.show()
    except Exception as exc:
        print(f"Visualization error: {exc}")
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to programmatically explore the FAIR² Mass Spectrometry dataset:

- Loaded Croissant metadata and extracted data using each RecordSet or distribution by its `@id`.
- Reviewed available fields and columns by their semantic IDs (`@id`).
- Performed cleaning, filtering, normalization, and grouping on numeric fields.
- Visualized data distributions using standard Python libraries.

This template can be adapted for any Croissant-compliant dataset. Ensure to reference fields, record sets, and distributions by their `@id` for reproducible, schema-compliant data science workflows.